In [ ]:
"""
Train a Liver Disease prediction model.

Dataset: Indian Liver Patient Dataset (ILPD) - UCI Machine Learning Repository
583 patient records, 10 features + 1 target column.

Features:
    Age
    Gender                        (Male=1, Female=0)
    Total_Bilirubin
    Direct_Bilirubin
    Alkaline_Phosphotase
    Alamine_Aminotransferase
    Aspartate_Aminotransferase
    Total_Protiens
    Albumin
    Albumin_and_Globulin_Ratio

Target: Dataset -> 1 = liver disease, 2 = no liver disease (original encoding)
        remapped here to 1 = liver disease, 0 = no liver disease
"""

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# ---------------------------------------------------------------------------
# Load & clean data
# ---------------------------------------------------------------------------
cols = ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin',
        'Alkaline_Phosphotase', 'Alamine_Aminotransferase',
        'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin',
        'Albumin_and_Globulin_Ratio', 'Dataset']

In [ ]:
df = pd.read_csv('liver_disease.csv', header=None, names=cols)

In [ ]:
# encode gender as numeric so the model + Streamlit text_input stay consistent
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [ ]:
# a handful of rows are missing Albumin_and_Globulin_Ratio - fill with mean
df['Albumin_and_Globulin_Ratio'] = df['Albumin_and_Globulin_Ratio'].fillna(
    df['Albumin_and_Globulin_Ratio'].mean()
)

In [ ]:
# remap target: 1 (disease) stays 1, 2 (no disease) -> 0
df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})

In [ ]:
feature_cols = cols[:-1]
X = df[feature_cols]
Y = df['Dataset'].astype(int)

In [ ]:
# ---------------------------------------------------------------------------
# Train / test split
# ---------------------------------------------------------------------------
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=2
)

In [ ]:
# ---------------------------------------------------------------------------
# Scale features
#
# Unlike the other 3 disease models in this project, Liver needed a change
# of approach (see model choice note below), and Logistic Regression is
# sensitive to feature scale (e.g. Alkaline_Phosphotase ranges into the
# thousands while Albumin_and_Globulin_Ratio is under 3).
# ---------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ---------------------------------------------------------------------------
# Train model
#
# WHY NOT THE LINEAR SVM USED FOR THE OTHER DISEASES?
# This dataset is imbalanced (416 disease vs 167 no-disease records, ~71/29).
# A plain linear SVM here just learns to predict "Disease" for almost every
# patient - it looks good on raw accuracy (~70%) purely because ~70% of the
# dataset has the disease, but it fails to identify any healthy patient
# (0% recall on the "No Disease" class - see the original notebook version).
#
# We compared several balanced approaches (SVM with class_weight='balanced',
# Random Forest, Gradient Boosting, SMOTE oversampling) using 5-fold
# cross-validation. Logistic Regression with class_weight='balanced' gave
# the best and most stable macro-F1 score, so we use it here instead.
# class_weight='balanced' automatically penalizes mistakes on the minority
# class (No Disease) more heavily during training, which stops the model
# from ignoring it.
# ---------------------------------------------------------------------------
classifier = LogisticRegression(class_weight='balanced', max_iter=2000)
classifier.fit(X_train_scaled, Y_train)

In [ ]:
train_acc = accuracy_score(Y_train, classifier.predict(X_train_scaled))
test_acc = accuracy_score(Y_test, classifier.predict(X_test_scaled))
test_pred = classifier.predict(X_test_scaled)
print('Liver Disease Model')
print('Accuracy score of the training data :', train_acc)
print('Accuracy score of the test data     :', test_acc)
print()
print('Confusion matrix (rows=actual, cols=predicted, 0=No Disease, 1=Disease):')
print(confusion_matrix(Y_test, test_pred))
print()
print(classification_report(Y_test, test_pred, target_names=['No Disease', 'Disease']))
print('Note: overall accuracy alone is misleading on this dataset because it is')
print('imbalanced (~70% of patients have the disease). Check the confusion matrix')
print('and per-class recall above to confirm the model is not just predicting')
print('"Disease" for everyone.')

In [ ]:
# ---------------------------------------------------------------------------
# Save model AND scaler
#
# The Streamlit app must scale new patient inputs the same way before
# calling .predict() - so the scaler has to be saved and shipped alongside
# the model, not just the model by itself.
# ---------------------------------------------------------------------------
filename = 'liver_disease_model.sav'
scaler_filename = 'liver_disease_scaler.sav'
pickle.dump(classifier, open(filename, 'wb'))
pickle.dump(scaler, open(scaler_filename, 'wb'))
print(f"Saved model to {filename}")
print(f"Saved scaler to {scaler_filename}")

In [ ]:
# quick sanity check reload
loaded_model = pickle.load(open(filename, 'rb'))
loaded_scaler = pickle.load(open(scaler_filename, 'rb'))
sample = np.asarray(X_test.iloc[0]).reshape(1, -1)
sample_scaled = loaded_scaler.transform(sample)
print('Sample prediction:', loaded_model.predict(sample_scaled), 'true label:', Y_test.iloc[0])